# record

> Asking a model once, and replaying the answer forever.

A test suite that calls a real model is slow, costs money, and fails for reasons that have nothing to do with the code. `CachedChat` runs each ask for real the first time, records what came back, and replays it after that — so the suite is fast and deterministic, and re-recording is one environment variable away.

::: {.callout-note}
This module is a candidate for extraction: it is a testing tool that happens to live next to the thing it tests. It needs `diskcache`, which is the `record` extra rather than a dependency.
:::

In [ ]:
#| default_exp record

In [ ]:
#| export
import json, os, re, shutil, tempfile
from contextlib import contextmanager
from hashlib import sha256
from fastcore.all import L, Path, store_attr, ifnone, listify, str2bool
from urai.core import Resp, resp_text
from urai.msgs import is_media, tc_name, est_tokens, render_prompt
from urai.caps import DFLT_CTX
from urai.opts import ChatOpts
from urai.chat import Chat
from urai.loop import ToolLoopMixin, UsageCallback, ToolReminderCallback, SlidingWindowCallback

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail
from urai.opts import RUNTIMES, ChatOpts, Runtime, register_runtime

## What is worth remembering

A backend that rejects its own tool call is a fact about the ask, and worth recording so the failure replays. A rate limit or a dropped socket is a fact about the afternoon; recording one would replay a bad afternoon forever, so those are raised and forgotten.

In [ ]:
#| export
CHAT_CACHE = 'chatcache'   #: default diskcache directory
KEY_VERSION = 3            #: part of every key; bump to invalidate every recording everywhere
RECORD_ENV = 'URAI_RECORD_CHAT'   #: asked when no caller said whether a miss may reach a model

_transient_re = re.compile(
    r'rate.?limit|timed? ?out|timeout|temporarily|try again|50[234]\b|429', re.I)

def is_transient(e):
    "Is `e` a failure of the moment rather than of the ask? Those are not worth remembering."
    return isinstance(e, (ConnectionError, TimeoutError, OSError)) or bool(_transient_re.search(str(e)))

In [ ]:
test_eq(is_transient(RuntimeError('429 rate limit')), True)
test_eq(is_transient(TimeoutError()), True)
test_eq(is_transient(ConnectionError('reset')), True)
test_eq(is_transient(RuntimeError('503 Service Unavailable')), True)
test_eq(is_transient(ValueError('unknown tool: frobnicate')), False)   # a fact about the ask

## The cache

`record` decides what a miss does. Left unset it reads the environment, and it reads it as a boolean — `URAI_RECORD_CHAT=0` has to mean no, or a CI job setting it to forbid live calls would be inviting them.

In [ ]:
#| export
class RecordCache:
    "Record what a call returned the first time and replay it after. The primitive under `CachedChat`."
    def __init__(self,
                 path=None,     # the diskcache directory; None -> `CHAT_CACHE`
                 record=None,   # let a miss run for real; None -> read `env`
                 env=RECORD_ENV,
                 version=None): # part of every key; None -> `KEY_VERSION`
        try: from diskcache import Cache
        except ImportError as e: raise ImportError(
            "RecordCache needs diskcache, which urai does not install by default: "
            "pip install 'urai[record]'") from e
        self.cache, self.env = Cache(str(path or CHAT_CACHE)), env
        self.version = ifnone(version, KEY_VERSION)
        self.record = str2bool(os.getenv(env) or '') if record is None else record

    def key(self, *parts):
        "Stable hash of `parts`, with the cache version in it."
        return sha256(json.dumps([self.version, *parts], sort_keys=True,
                                 default=str).encode()).hexdigest()

    def __call__(self, key, f, what=''):
        "Replay `key`, else run `f()` and record what it did, including how it failed."
        if key in self.cache:
            got, val = self.cache[key]
            if got == 'exc': raise RuntimeError(val)
            return val
        if not self.record: raise KeyError(
            f'no recording for {what or key[:16]} - set {self.env}=1 and re-run to record it')
        try: val = f()
        except Exception as e:
            if not is_transient(e): self.cache[key] = ('exc', f'{type(e).__name__}: {e}')
            raise
        self.cache[key] = ('ok', val)
        return val

    def forget(self, key):
        "Drop one recording, for when what it captured was never the truth. Was there one?"
        return self.cache.pop(key, None) is not None

In [ ]:
d = tempfile.mkdtemp()
rec = RecordCache(d, record=True)
calls = []
test_eq(rec(rec.key('a'), lambda: calls.append(1) or 'first'), 'first')
test_eq(rec(rec.key('a'), lambda: calls.append(1) or 'second'), 'first')   # replayed
test_eq(len(calls), 1)

In [ ]:
test_eq(rec.key('a') == rec.key('a'), True)          # stable
test_eq(rec.key('a') == rec.key('b'), False)
test_eq(RecordCache(d, record=True, version=99).key('a') == rec.key('a'), False)  # version is in it

In [ ]:
# a failure about the ask is recorded and replays; a failure about the afternoon is not
test_fail(lambda: rec(rec.key('bad'), lambda: (_ for _ in ()).throw(ValueError('no such tool'))))
test_fail(lambda: rec(rec.key('bad'), lambda: 'never runs'), contains='ValueError: no such tool')

In [ ]:
test_fail(lambda: rec(rec.key('flaky'), lambda: (_ for _ in ()).throw(TimeoutError('429'))))
test_eq(rec.key('flaky') in rec.cache, False)        # forgotten, so the next run may try again

In [ ]:
# with recording off, a miss is an error that says how to fix itself
ro = RecordCache(d, record=False)
test_fail(lambda: ro(ro.key('unseen'), lambda: 'x'), contains='URAI_RECORD_CHAT=1')
test_eq(ro(rec.key('a'), lambda: 'x'), 'first')      # ...but a hit still replays

In [ ]:
test_eq(rec.forget(rec.key('a')), True)
test_eq(rec.forget(rec.key('a')), False)             # already gone

In [ ]:
os.environ['URAI_RECORD_CHAT'] = '0'
test_eq(RecordCache(d).record, False)                # '0' means no, not "a non-empty string"
os.environ['URAI_RECORD_CHAT'] = '1'
test_eq(RecordCache(d).record, True)
del os.environ['URAI_RECORD_CHAT']
test_eq(RecordCache(d).record, False)

## Replaying without writing to what you replay from

Opening a diskcache writes to it — SQLite bookkeeping, not content — so a cache committed to a repository comes back modified after every run that reads it, and every diff carries it. That defeats committing it, which is the whole point of a recording.

`replaying` hands back the directory to open and whether it may record, as one pair. When there is something to record that is the real directory; when there is not it is a throwaway copy, and the committed one is left byte for byte as it was.

The two answers come back together because they have to agree. Resolved separately — the directory from one reading of the environment and the cache's `record` from another — they drift, and the shape of the drift is nasty in both directions: a replay that writes to the artefact it was protecting, or a recording that files everything into a copy about to be deleted.

In [ ]:
#| export
@contextmanager
def replaying(path=None,        # the diskcache directory; None -> `CHAT_CACHE`
              record=None,      # let a miss reach a real model; None -> read `env`
              env=RECORD_ENV):  # which variable that is
    """`(directory, record)` for a `RecordCache`: the real one to record into, a copy to replay from.

    Yield both, never one: which directory this is and whether the cache in it may write are the
    same question, and answering it twice is how a replay ends up writing to the artefact it was
    supposed to leave alone -- or a recording ends up in a copy that is about to be deleted."""
    rec = str2bool(os.getenv(env) or '') if record is None else bool(record)
    p = Path(path or CHAT_CACHE)
    if rec or not p.exists():
        p.mkdir(parents=True, exist_ok=True)
        yield p, rec
        return
    d = Path(tempfile.mkdtemp())/'chats'
    shutil.copytree(p, d)
    try: yield d, rec
    finally: shutil.rmtree(d, ignore_errors=True)

In [ ]:
src = Path(tempfile.mkdtemp())/'committed'
rc = RecordCache(src, record=True); rc(rc.key('q'), lambda: 'an answer'); rc.cache.close()
before = (src/'cache.db').read_bytes()

# replaying reads a copy, and the committed cache is untouched afterwards
with replaying(src) as (d, rec):
    test_eq(rec, False)
    assert d != src, 'a replay reads a copy'
    test_eq(RecordCache(d, record=False)(RecordCache(d).key('q'), None), 'an answer')
test_eq((src/'cache.db').read_bytes(), before)   # ...byte for byte

# recording writes to the real one, because a recording filed into a copy is a recording lost
with replaying(src, record=True) as (d, rec): test_eq((d, rec), (src, True))

# and the environment decides when the caller did not, for both halves at once
os.environ['URAI_RECORD_CHAT'] = '1'
with replaying(src) as (d, rec): test_eq((d, rec), (src, True))
os.environ['URAI_RECORD_CHAT'] = '0'
with replaying(src) as (d, rec): test_eq((d != src, rec), (True, False))

# a directory that is not there yet is made, not copied: there is nothing to protect
fresh = Path(tempfile.mkdtemp())/'new'
with replaying(fresh) as (d, rec): test_eq((d, d.exists()), (fresh, True))

## The chat

What goes into a key is everything a reply depends on: the model, how it was built, the system prompt, the tool names, and the conversation so far. Media is hashed rather than stored, so a key stays small.

In [ ]:
#| export
def canon_msg(m):
    "What a reply depends on in one history entry: who spoke, what was said, what was attached, what was called."
    media = [sha256(str(p).encode()).hexdigest()[:16] for p in listify(m.get('content')) if is_media(p)]
    tcs = [(tc_name(tc), (tc.get('function') or {}).get('arguments')) for tc in (m.get('tool_calls') or [])]
    return [m.get('role', ''), resp_text(m), media, tcs]

In [ ]:
test_eq(canon_msg({'role': 'user', 'content': 'hi'}), ['user', 'hi', [], []])
from urai.msgs import ToolCall, mk_msg
test_eq(canon_msg({'role': 'assistant', 'content': '', 'tool_calls': [ToolCall('add', {'a': 1})]}),
        ['assistant', '', [], [('add', {'a': 1})]])

In [ ]:
png = b'\x89PNG\r\n\x1a\n' + b'\x00' * 40
m = canon_msg(mk_msg(['look', png]))
test_eq((m[0], m[1]), ('user', 'look'))
test_eq(len(m[2][0]), 16)              # the picture is hashed, not carried into the key

A recorded turn keeps the whole slice of history it produced, not just the reply. A tool-using turn is a call, a result and an answer; a recording that kept only the answer would replay a conversation the model was never in — and since that same history is what the *next* key is built from, it would also make two different tool conversations with the same prompt collide on one key.

In [ ]:
#| export
class CachedChat:
    "A `Chat` whose replies are recorded to disk and replayed on a second ask. A replay builds no engine."
    def __init__(self,
                 model=None,   # anything `Chat` takes; part of the key
                 path=None,    # the diskcache directory; None -> `CHAT_CACHE`
                 record=None,  # let a miss reach a real model; None -> `$URAI_RECORD_CHAT`
                 sp='',        # system prompt, part of the key
                 tools=None,   # tool *names* are part of the key; the real chat gets the tools
                 **kw):        # forwarded to `Chat` on a miss
        store_attr('model,sp,kw')
        self.tools, self.rec, self._chat, self.hist = L(tools), RecordCache(path, record), None, []

    @property
    def cache(self): return self.rec.cache

    def cancel(self):
        "Stop the live turn, if one was ever built. Nothing to stop while replaying."
        return self._chat.cancel() if self._chat is not None else False

    @property
    def cancelled(self): return self._chat is not None and self._chat.cancelled

    @property
    def chat(self):
        "The real `Chat`, built only when something actually has to be asked."
        if self._chat is None:
            self._chat = Chat(self.model, sp=self.sp, tools=list(self.tools),
                              messages=self.hist, **self.kw)
        return self._chat

    def _key(self, kind, *args, hist=True):
        "What a reply is recorded under. `hist=False` is for asks that are stateless by definition."
        return self.rec.key(self.model, kind, self.sp, self.kw,
                            [getattr(t, '__name__', str(t)) for t in self.tools],
                            [canon_msg(m) for m in self.hist] if hist else [], *args)

    def _ask(self, kind, args, f, hist=True):
        "Replay this ask if it is recorded, else run `f()` and record it."
        n = len(self.hist) if hist else 0
        what = f'{kind} for {self.model} after {n} messages: {str(args[0])[:80]}'
        return self.rec(self._key(kind, *args, hist=hist), f, what)

    def __call__(self, prompt, **kw):
        "One turn, replayed if it has been asked before. `hist` ends up as the real turn left it."
        key = self._key('call', prompt, kw)
        hit = key in self.rec.cache
        what = f'call for {self.model} after {len(self.hist)} messages: {str(prompt)[:80]}'
        def _live():
            n = len(self.chat.hist)
            r = self.chat(prompt, **kw)
            return {'resp': dict(r), 'hist': [dict(m) for m in self.chat.hist[n:]]}
        rec = self.rec(key, _live, what)
        # a stopped turn is a truncated one, and a recording would replay the truncation forever
        if not hit and self.cancelled: self.rec.forget(key)
        self.hist += [dict(m) for m in rec['hist']]
        # a replayed turn never reached the live chat, so hand it the history it missed
        if hit and self._chat is not None:
            self._chat.hist = list(self.hist)
            self._chat._recreate_conv()
        return Resp(rec['resp'])

    def oneshot(self, prompt, sp='', think=None, max_tokens=None):
        return self._ask('oneshot', [prompt, sp, think, max_tokens], hist=False,
                         f=lambda: self.chat.oneshot(prompt, sp, think=think, max_tokens=max_tokens))

    def classify(self, text, labels, **kw):
        return self._ask('classify', [text, list(labels), kw], hist=False,
                         f=lambda: self.chat.classify(text, labels, **kw))

    def reconfigure(self, sp=None, tools=None):
        "Change `sp` or `tools` for the asks that follow. Both are in the key, so a replay stays honest."
        if sp is not None: self.sp = sp
        if tools is not None: self.tools = L(tools)
        if self._chat is not None: self._chat.reconfigure(sp=sp, tools=tools)
        return self

    def close(self):
        if self._chat is not None: self._chat.close(); self._chat = None

In [ ]:
built = []

class _CountChat(Chat):
    "Counts how often a real chat was built and asked, so replays can be told from live calls."
    _runtime, ctx_limit, token_count = 'count', 8192, 0
    def __init__(self, model=None, **kw):
        built.append(model)
        self._setup(model, ChatOpts.create(kw.pop('opts', None), **kw))
    def _send(self, msg, **kw):
        self.hist.append(self.mk_msg(msg))
        self.turn_res = Resp({'role': 'assistant', 'content': f'reply to {msg}'})
        self.hist.append(dict(self.turn_res))
        return self.turn_res
    def _oneshot(self, prompt, sp='', think=None, max_tokens=None): return f'oneshot: {prompt}'

register_runtime(Runtime('count', _CountChat, ('count-',)))

In [ ]:
d = tempfile.mkdtemp()
c = CachedChat('count-1', path=d, record=True, runtime='count')
test_eq(resp_text(c('hello')), 'reply to hello')
test_eq(built, ['count-1'])                # built once, on the first miss

In [ ]:
built.clear()
c2 = CachedChat('count-1', path=d, record=False, runtime='count')
test_eq(resp_text(c2('hello')), 'reply to hello')
test_eq(built, [])                         # replayed, so no chat was ever built
test_eq([m['role'] for m in c2.hist], ['user', 'assistant'])   # ...and the history came back

In [ ]:
# the conversation so far is part of the key, so the same prompt twice is two recordings
test_fail(lambda: c2('hello'), contains='no recording')
test_eq(resp_text(c('hello')), 'reply to hello')   # the recorder can, and records the second turn
test_eq(resp_text(c2('hello')), 'reply to hello')  # ...so now the replayer can too

In [ ]:
c3 = CachedChat('count-1', path=d, record=False, sp='a different briefing', runtime='count')
test_fail(lambda: c3('hello'), contains='no recording')   # sp is part of the key

In [ ]:
built.clear()
c4 = CachedChat('count-1', path=d, record=True, runtime='count')
test_eq(c4.oneshot('what is 2+2?'), 'oneshot: what is 2+2?')
c4('hello')                                          # a turn that changes the history
test_eq(c4.oneshot('what is 2+2?'), 'oneshot: what is 2+2?')
test_eq(len([b for b in built]), 1)                  # one chat; the second oneshot replayed

In [ ]:
test_eq(c4.reconfigure(sp='new').sp, 'new')
test_eq(c4.cancelled, False)
c4.close()
test_eq(c4.cancel(), False)                # nothing live to stop

## A recording a harness can drive

`CachedChat` wraps a `Chat` rather than being one. It carries its own `hist` and forwards four methods, so it has no `use`, no `ctx_limit`, no `token_count`, no `mk_msgs` and no tool loop, and anything built on `Chat` cannot drive one. That is why it appears above and in nothing downstream.

`RecordedChat` *is* a `Chat`. The recording sits at `_model_step`, the lowest seam there is: one entry per wire call, keyed by the conversation that produced it. Everything above stays the real thing — the tool loop runs, the approval gate fires, the budget counts, usage folds, the sliding window evicts. A replay builds no engine and opens no socket; the live chat is constructed only on the first miss.

Recording at the step rather than at the turn is what makes a tool-calling conversation replayable. The model's calls come back from the recording and the tools themselves really run, so a test about what a tool did is still a test about the tool.

In [ ]:
#| export
class RecordedChat(ToolLoopMixin, Chat):
    "A real `Chat` whose wire calls are recorded once and replayed after. See `CachedChat` for the wrapper it replaces."

    _runtime = 'recorded'
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]

    def __new__(cls, *args, **kw): return object.__new__(cls)   # never re-routed by runtime

    def __init__(self,
                 model=None,        # anything `Chat` takes; part of every key
                 *,
                 path=None,         # the diskcache directory; None -> `CHAT_CACHE`
                 record=None,       # let a miss reach a real model; None -> read `env`
                 env=None,          # which variable that is; None -> `RECORD_ENV`
                 runtime=None,      # forced runtime for the live chat
                 model_path=None,
                 opts=None,
                 ctx_limit=None,
                 **kw):             # the live chat's own constructor arguments
        o = ChatOpts.create(opts, **({'ctx_limit': ctx_limit} if ctx_limit else {}), **kw)
        self.model, self.model_path, self._rt = model, model_path, runtime
        # the loose keywords, kept as they arrived: a backend's own constructor arguments
        # (`workspace`, `engine`, `quant`) are named parameters there, not portable options,
        # and `ChatOpts` would carry them into `extra` and hand them to the wire instead
        self._kw = {**({'ctx_limit': ctx_limit} if ctx_limit else {}), **kw}
        self.rec = RecordCache(path, record, env=env or RECORD_ENV)
        self._chat, self._ctx_tokens = None, 0
        self._set_tools(o.tools)               # `toolspecs` and `ns`: `_setup` sets `tools` alone
        self.ctx_limit = o.ctx or DFLT_CTX
        self._setup(model, o)

    @property
    def chat(self):
        "The live `Chat`, built on the first miss. A full replay never touches this."
        if self._chat is None:
            self._chat = Chat(self.model, runtime=self._rt, model_path=self.model_path, **self._kw)
        return self._chat

    def _live(self):
        "The live chat, holding exactly the conversation this one holds."
        c = self.chat
        c.hist = [dict(m) for m in self.hist]
        c._recreate_conv()
        return c

    def _key(self, kind, *args, hist=True):
        "What one call is recorded under: the model, the briefing, the tools, and the conversation."
        return self.rec.key(self.model, self._rt, kind, self.sp,
                            [getattr(t, '__name__', str(t)) for t in self.tools],
                            [canon_msg(m) for m in self.hist] if hist else [], *args)

    def _what(self, kind):
        last = resp_text(self.hist[-1]) if self.hist else ''
        return f'{kind} for {self.model} after {len(self.hist)} messages: {last[:80]}'

    def _model_step(self, max_output_tokens=None, **kw):
        res = self.rec(self._key('step', kw), lambda: dict(self._live()._model_step(**kw)),
                       self._what('step'))
        return Resp(res)

    def _stream_step(self, max_output_tokens=None, **kw):
        def live():
            c = self._live()
            chunks = [o for o in c._stream_step(**kw)]
            return {'chunks': chunks, 'res': dict(c._step_res)}
        rec = self.rec(self._key('stream', kw), live, self._what('stream'))
        yield from rec['chunks']
        self._step_res = Resp(rec['res'])

    def _oneshot(self, prompt, sp='', think=None, max_tokens=None):
        return self.rec(self._key('oneshot', prompt, sp, think, max_tokens, hist=False),
                        lambda: self.chat.oneshot(prompt, sp, think=think, max_tokens=max_tokens),
                        f'oneshot for {self.model}: {str(prompt)[:80]}')

    @property
    def token_count(self):
        "The live chat's count where there is one, else the estimate every backend falls back to."
        if self._chat is not None: return self._chat.token_count
        return est_tokens(render_prompt(self.hist)) + est_tokens(self.sp)

    def count_tokens(self, text):
        f = getattr(self._chat, 'count_tokens', None)
        return f(text) if f else est_tokens(str(text or ''))

    def close(self):
        if self._chat is not None: self._chat.close(); self._chat = None

In [ ]:
#| hide
from urai.core import UsageStats
from urai.msgs import ToolCall

wire = []

class _StepChat(ToolLoopMixin, Chat):
    "A chat whose wire call is a script, so a replay can be told from a live call."
    _runtime, ctx_limit, token_count = 'step', 8192, 0
    _USE = {'prompt_tokens': 10, 'completion_tokens': 5, 'total_tokens': 15, 'n': 1, 'model': 'scripted'}

    def __init__(self, model=None, **kw):
        wire.append(('built', model))
        self._ctx_tokens = 0
        o = ChatOpts.create(kw.pop('opts', None), **kw)
        self._set_tools(o.tools)
        self._setup(model, o)

    def _model_step(self, **kw):
        wire.append(('step', len(self.hist)))
        last = resp_text(self.hist[-1])
        if 'plus' in last and any(t.__name__ == 'add' for t in self.tools):
            return Resp({'role': 'assistant', 'content': '', 'usage': self._USE,
                         'tool_calls': [ToolCall('add', {'a': 17, 'b': 25}, id='c1')]})
        return Resp({'role': 'assistant', 'content': f'reply to {last}', 'usage': self._USE})

    def _stream_step(self, **kw):
        wire.append(('stream', len(self.hist)))
        last = resp_text(self.hist[-1])
        for w in ('reply ', 'to ', last): yield {'content': w}
        self._step_res = Resp({'role': 'assistant', 'content': f'reply to {last}', 'usage': self._USE})

register_runtime(Runtime('step', _StepChat, ('step-',)))

In [ ]:
# a `RecordedChat` is a `Chat`, which is the whole difference: a harness can drive one
d = tempfile.mkdtemp()
r = RecordedChat('step-1', path=d, record=False, runtime='step')
test_eq(isinstance(r, Chat), True)
for name in ('use', 'hist', 'sp', 'approve', 'ctx_limit', 'token_count', 'count_tokens',
             'mk_msgs', 'pct_full', 'max_steps', 'tool_max_len', 'toolspecs'):
    assert hasattr(r, name), name
test_eq(r.hist, [])
test_eq(r._chat, None)                       # nothing built, because nothing has been asked

# a miss with no recording names the ask and says how to record it, rather than reaching a model
test_fail(lambda: r('hello'), contains='no recording for')
test_eq(r._chat, None)                       # ...and still nothing built
test_eq(wire, [])                            # ...and no wire call was made

# the key is the conversation, so two conversations are two recordings, and the same one is one
r.hist.clear()                               # the failed ask above left its question behind
empty = r._key('step', {})
r.hist.append({'role': 'user', 'content': 'hello'})
assert empty != r._key('step', {})
r.hist.clear()
test_eq(empty, r._key('step', {}))

In [ ]:
wire.clear()
live = RecordedChat('step-1', path=d, record=True, runtime='step')
test_eq(resp_text(live('hello')), 'reply to hello')
test_eq(wire, [('built', 'step-1'), ('step', 1)])       # one chat built, one wire call
test_eq(live.use.total_tokens, 15)                      # usage folded through the real callback

wire.clear()
replay = RecordedChat('step-1', path=d, record=False, runtime='step')
test_eq(resp_text(replay('hello')), 'reply to hello')
test_eq(wire, [])                                       # nothing built, nothing asked
test_eq([m['role'] for m in replay.hist], ['user', 'assistant'])
test_eq(replay.use.total_tokens, 15)                    # ...and the turn is still charged

In [ ]:
ran = []
def add(a: int, b: int) -> int:
    "Add two whole numbers."
    ran.append((a, b)); return a + b

wire.clear()
live = RecordedChat('step-1', path=d, record=True, runtime='step', tools=[add], max_steps=3)
test_eq(resp_text(live('what is 17 plus 25')), 'reply to 42')
test_eq(ran, [(17, 25)])
test_eq([m['role'] for m in live.hist], ['user', 'assistant', 'tool', 'assistant'])

# the model's decision is replayed; the tool is not. It runs again, here, now -- which is what
# makes an assertion about what a tool did worth making
ran.clear(); wire.clear()
replay = RecordedChat('step-1', path=d, record=False, runtime='step', tools=[add], max_steps=3)
test_eq(resp_text(replay('what is 17 plus 25')), 'reply to 42')
test_eq(ran, [(17, 25)])                   # the tool ran here, with the arguments the model chose
test_eq(wire, [])
test_eq(replay.hist[2]['content'], '42')

In [ ]:
# a stream replays in pieces, and lands the same answer
wire.clear()
test_eq(''.join(RecordedChat('step-1', path=d, record=True, runtime='step')('hi', stream=True)), 'reply to hi')
test_eq(wire, [('built', 'step-1'), ('stream', 1)])
wire.clear()
chunks = list(RecordedChat('step-1', path=d, record=False, runtime='step')('hi', stream=True))
test_eq(len(chunks) > 1, True)                # a stream that arrives in one piece is not a stream
test_eq(''.join(chunks), 'reply to hi')
test_eq(wire, [])

# a one-shot has no history by definition, so its recording is keyed without one
test_fail(lambda: replay.oneshot('anything'), contains='no recording for oneshot')

# tools are part of the key, so the same prompt with a different surface is a different recording
test_fail(lambda: RecordedChat('step-1', path=d, record=False, runtime='step',
                               tools=[add])('hello'), contains='no recording')

In [ ]:
#| hide
del RUNTIMES['step']

In [ ]:
#| hide
del RUNTIMES['count']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()